# GPT Goated

In [4]:
!pip install transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 221.1 kB/s eta 0:00:0000:0100:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.2/297.2 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 456.9 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.5/416.5 kB 184.1 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 56.3 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.0/143.0 kB 42.9 kB/s eta 0:00:00a 0:00:01


# 2. Data Preprocessing

In [6]:
# 2.1 Load ESM-2 Model and Tokenizer
import torch
from transformers import EsmTokenizer, EsmModel

# Load the tokenizer and model for ESM-2
tokenizer = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
model = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D")

Some weights of the model checkpoint at facebook/esm2_t6_8M_UR50D were not used when initializing EsmModel: ['lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.dense.bias']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# 2.2 Tokenize and Embed Protein Sequences
def get_protein_embeddings(sequence):
    inputs = tokenizer(sequence, return_tensors="pt", add_special_tokens=False)
    with torch.no_grad():
        outputs = model(**inputs)
    # Get the embeddings from the last hidden state
    embeddings = outputs.last_hidden_state
    return embeddings.squeeze(0).mean(dim=0)  # Mean pooling over the sequence length

# Example sequence
sequence = "MKQLEDKVEELLSKNYHLENEVARLKKLV"
embedding = get_protein_embeddings(sequence)
print(embedding.shape)  # Should be (1280,) for ESM-2 model


torch.Size([320])


In [ ]:
# 2.3 Prepare Dataset --> converting  protein sequences into embeddings
import pandas as pd

# Assuming you have a DataFrame with sequences and labels
df = pd.read_csv("your_dataset.csv")

# Apply the embedding function to all sequences
df['embedding'] = df['sequence'].apply(get_protein_embeddings)

# Convert the embeddings and labels to tensors
X = torch.stack(df['embedding'].tolist())
y = torch.tensor(df['label'].values)


In [ ]:
# 2.4 Train a Classifier on the Embeddings
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

class ProteinFunctionClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(ProteinFunctionClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

# Create the dataset and dataloader
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Instantiate the model, loss function, and optimizer
model = ProteinFunctionClassifier(input_dim=embedding.shape[0], num_classes=y.max().item() + 1)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y.float())
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}")

# Save the trained model
torch.save(model.state_dict(), "protein_function_classifier.pth")


In [ ]:
# 2.5 Evaluate Model
from sklearn.metrics import f1_score, precision_recall_curve, auc

# Make predictions
model.eval()
with torch.no_grad():
    y_pred = model(X).numpy()

# Calculate metrics
f1 = f1_score(y, y_pred.round(), average='micro')
precision, recall, _ = precision_recall_curve(y.flatten(), y_pred.flatten())
aupr = auc(recall, precision)

print(f"F1 Score: {f1}")
print(f"AUPR: {aupr}")